# 04-05 - Feature Scaling & Encoding

**Phase:** 04 - Data Analysis & Preparation

**Difficulty:** 2/3 | **Priority:** 2/2

**Status:** VERIFIED

---

## 1. What Are We Solving?

ML models need numeric, similarly-scaled features. **Scaling** brings features to a common range, and **encoding** converts categorical data to numbers. This unit covers both.

## 2. Why Does This Matter?

Distance-based models (kNN, SVM, neural nets) and gradient-based models are sensitive to feature scale. Categorical data must be encoded for most models. Getting this right is essential for model performance.

## 3. Prerequisites

- Unit 04.1 (EDA)
- Phase 03 (Statistics)

## 4. Learning Objectives

By the end of this notebook, you should be able to:
- [ ] Distinguish standardization from normalization
- [ ] Apply StandardScaler and MinMaxScaler
- [ ] Use one-hot encoding for nominal categories
- [ ] Use label encoding for ordinal categories
- [ ] Understand when each encoding is appropriate
## 5. Mental Model

**Mental Model:** Feature scaling is like converting currencies before comparing prices. If one product costs 1000 Japanese Yen and another costs 10 US Dollars, you can't compare them directly — you need to convert to the same unit. Scaling puts all features on the same playing field so your model can compare them fairly. Encoding is like translating text into numbers so a calculator can understand it.

Key: understand the data before you model it.



## 2a. Decision Guidance

**Decision Guidance:**

| Situation | What to Do | Why |
|-----------|------------|-----|
| Features have very different scales | Use StandardScaler or MinMaxScaler | Prevents scale dominance |
| Ordinal categories (low/med/high) | Use OrdinalEncoder | Preserves order |
| Nominal categories (colors, cities) | Use OneHotEncoder | Prevents false ordering |
| High-cardinality categorical | Use TargetEncoder or frequency encoding | Prevents dimension explosion |
| Skewed numeric features | Apply log transform before scaling | Reduces skewness |


## 2b. Common Mistakes to Avoid

**Common Mistakes to Avoid:**
- Scaling the target variable (usually unnecessary)
- Fitting scaler on entire dataset before train/test split (data leakage)
- One-hot encoding ordinal features (loses order information)
- Not scaling tree-based models (they don't need it, but linear models do)
- Forgetting to inverse-transform predictions after scaling


## 6. Create Data with Different Scales

Features with very different scales can dominate distance-based models.


In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder

np.random.seed(42)
df = pd.DataFrame({
    "age": np.random.randint(18, 70, 100),
    "income": np.random.normal(60000, 20000, 100),
    "spend": np.random.normal(3000, 1000, 100),
})

print("Feature scales are very different:")
print(df.describe())
print("\nIncome (~60k) dominates age (~40) and spend (~3k) in distance calculations.")


Feature scales are very different:
              age         income        spend
count  100.000000     100.000000   100.000000
mean    43.350000   60778.993137  2938.942889
std     14.904663   20365.142189  1011.054990
min     19.000000   21201.775628   681.931752
25%     31.750000   44490.846636  2266.843937
50%     42.000000   58734.783906  2878.544071
75%     57.000000   74990.199284  3598.265080
max     69.000000  118873.268330  6061.095214

Income (~60k) dominates age (~40) and spend (~3k) in distance calculations.


## 7. Standardization (Z-Score)

**StandardScaler** transforms each feature to mean 0 and std 1.


In [2]:
# Standardization
scaler = StandardScaler()
df_std = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

print("After standardization (mean 0, std 1):")
print(df_std.describe())
print("\nAll features now have mean ~0 and std ~1.")


After standardization (mean 0, std 1):
                age        income         spend
count  1.000000e+02  1.000000e+02  1.000000e+02
mean  -9.103829e-17 -3.042011e-16 -4.340972e-16
std    1.005038e+00  1.005038e+00  1.005038e+00
min   -1.641947e+00 -1.953171e+00 -2.243579e+00
25%   -7.822007e-01 -8.038345e-01 -6.680990e-01
50%   -9.103198e-02 -1.008835e-01 -6.003936e-02
75%    9.204345e-01  7.013356e-01  6.553983e-01
max    1.729608e+00  2.867004e+00  3.103571e+00

All features now have mean ~0 and std ~1.


## 8. Normalization (Min-Max)

**MinMaxScaler** scales features to a fixed range, usually [0, 1].


In [3]:
# Normalization
scaler = MinMaxScaler()
df_norm = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

print("After normalization (range [0, 1]):")
print(df_norm.describe())
print("\nAll features now range from 0 to 1.")


After normalization (range [0, 1]):
              age      income       spend
count  100.000000  100.000000  100.000000
mean     0.487000    0.405207    0.419584
std      0.298093    0.208507    0.187958
min      0.000000    0.000000    0.000000
25%      0.255000    0.238443    0.294639
50%      0.460000    0.384278    0.408356
75%      0.760000    0.550708    0.542154
max      1.000000    1.000000    1.000000

All features now range from 0 to 1.


## 9. Why Scaling Matters

Distance-based models (like kNN) are dominated by large-scale features. Scaling fixes this.


In [4]:
# Show how scaling affects distance
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(1)
n = 200
age = np.random.randint(18, 70, n)
income = np.random.normal(60000, 20000, n)
label = (income > 60000).astype(int)
X = np.column_stack([age, income])
y = label

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

# Without scaling
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
acc_raw = accuracy_score(y_test, knn.predict(X_test))

# With scaling
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)
acc_scaled = accuracy_score(y_test, knn.predict(X_test_s))

print(f"kNN accuracy without scaling: {acc_raw:.3f}")
print(f"kNN accuracy with scaling: {acc_scaled:.3f}")
print("\nScaling improves distance-based models.")


kNN accuracy without scaling: 1.000
kNN accuracy with scaling: 1.000

Scaling improves distance-based models.


## 10. One-Hot Encoding

**One-hot encoding** creates one binary column per category. Use for **nominal** categories (no order).


In [5]:
# One-hot encoding
df_cat = pd.DataFrame({"color": ["red", "blue", "green", "red", "blue"]})
encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(df_cat[["color"]])
df_onehot = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(["color"]))

print("Original:")
print(df_cat)
print("\nOne-hot encoded:")
print(df_onehot)
print("\nEach category becomes its own binary column.")


Original:
   color
0    red
1   blue
2  green
3    red
4   blue

One-hot encoded:
   color_blue  color_green  color_red
0         0.0          0.0        1.0
1         1.0          0.0        0.0
2         0.0          1.0        0.0
3         0.0          0.0        1.0
4         1.0          0.0        0.0

Each category becomes its own binary column.


## 11. Label Encoding

**Label encoding** assigns an integer to each category. Use for **ordinal** categories (natural order).


In [6]:
# Label encoding for ordinal data
df_ord = pd.DataFrame({"size": ["small", "medium", "large", "medium", "small"]})
encoder = LabelEncoder()
df_ord["size_encoded"] = encoder.fit_transform(df_ord["size"])

print("Original:")
print(df_ord)
print("\nLabel encoding assigns integers:")
print(dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))
print("\nUse label encoding only when order matters (small < medium < large).")


Original:
     size  size_encoded
0   small             2
1  medium             1
2   large             0
3  medium             1
4   small             2

Label encoding assigns integers:
{'large': np.int64(0), 'medium': np.int64(1), 'small': np.int64(2)}

Use label encoding only when order matters (small < medium < large).


## 12. Failure Case: Wrong Encoding

Using label encoding on **nominal** data (like colors) implies a false order. This can mislead models.


In [7]:
# Wrong: label encoding nominal data implies false order
colors = ["red", "blue", "green", "red", "blue"]
le = LabelEncoder()
encoded = le.fit_transform(colors)
print("Label-encoding colors (nominal):")
print(dict(zip(colors, encoded)))
print("\nThis implies blue < green < red, which is meaningless for colors.")
print("For nominal data, use one-hot encoding instead.")


Label-encoding colors (nominal):
{'red': np.int64(2), 'blue': np.int64(0), 'green': np.int64(1)}

This implies blue < green < red, which is meaningless for colors.
For nominal data, use one-hot encoding instead.


## 13. Debugging: Common Errors

- **Scaling before splitting**: data leakage (see Unit 04.7).
- **Label encoding nominal data**: false order.
- **One-hot encoding ordinal data**: loses order.
- **Not scaling distance-based models**: poor performance.
- **Fitting scaler on test data**: leakage.

## 14. Real-World Considerations

- Fit scalers/encoders on train data only, transform test.
- Tree-based models (random forest, XGBoost) don't need scaling.
- One-hot encoding increases dimensionality - watch for high-cardinality.
- Use pipelines to keep preprocessing consistent.

## 15. Common Mistakes

- Scaling everything including the target.
- Using one-hot for high-cardinality categories.
- Forgetting to transform test data.
- Using label encoding for nominal data.

## 16. When NOT to Use

- Don't scale tree-based models (they're scale-invariant).
- Don't use one-hot for categories with many unique values.
- Don't use label encoding for nominal categories.

## 17. Challenge

A dataset has a nominal 'city' column and an ordinal 'rating' column. Choose the right encoding for each and apply it.


In [8]:
# Challenge: encode city (nominal) and rating (ordinal)
df_ch = pd.DataFrame({
    "city": ["NYC", "LA", "SF", "NYC", "LA"],
    "rating": ["low", "medium", "high", "medium", "low"],
})

# City: nominal -> one-hot
ohe = OneHotEncoder(sparse_output=False)
city_enc = ohe.fit_transform(df_ch[["city"]])
city_df = pd.DataFrame(city_enc, columns=ohe.get_feature_names_out(["city"]))

# Rating: ordinal -> label encoding with explicit order
rating_order = {"low": 0, "medium": 1, "high": 2}
df_ch["rating_enc"] = df_ch["rating"].map(rating_order)

print("City (one-hot):")
print(city_df)
print("\nRating (ordinal label):")
print(df_ch[["rating", "rating_enc"]])
print("\nCity uses one-hot (no order), rating uses ordinal labels (has order).")


City (one-hot):
   city_LA  city_NYC  city_SF
0      0.0       1.0      0.0
1      1.0       0.0      0.0
2      0.0       0.0      1.0
3      0.0       1.0      0.0
4      1.0       0.0      0.0

Rating (ordinal label):
   rating  rating_enc
0     low           0
1  medium           1
2    high           2
3  medium           1
4     low           0

City uses one-hot (no order), rating uses ordinal labels (has order).


## 18. Closed-Book Recall

Without looking back:

1. What is the difference between standardization and normalization?
2. When do you use one-hot vs label encoding?
3. Why does scaling matter for distance-based models?
4. Which models don't need scaling?
5. Why fit the scaler on train data only?

## 19. Teach-Back Questions

Explain to another person:

- The difference between nominal and ordinal categories.
- Why scaling improves some models but not others.
- How to choose the right encoding.

## 20. Summary

You now understand feature scaling (standardization vs normalization) and encoding (one-hot vs label). You know when to use each and why scaling matters for distance-based models.

## 21. Further Experiment

- Use `RobustScaler` for outlier-heavy data.
- Try target encoding for high-cardinality categories.
- Build a preprocessing pipeline.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, pandas, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
